# Amazon ML 2026 — Colab T4

**Run all cells from top to bottom.** Select **Runtime → Change runtime type → T4 GPU** and **High-RAM** if available.

The dataset already lives on your Drive. Its path is filled below. Full mode processes all train/test files and produces both TSV outputs. The existing 30-million candidate-row training cap is retained.

Completed stages are copied to Drive. A reset can repeat the unfinished stage; there is no guarantee of uninterrupted Colab runtime. Allow substantial backup time and roughly **80–100 GB free on Drive**, plus **60 GiB local capacity** and **at least 28 GiB system RAM** for full mode. These are conservative guards, not measured guarantees.

This notebook uses an isolated Python 3.11 environment so the original pinned packages do not overwrite Colab's Python environment. **Do not run the old Plan2 notebook.**


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# Existing file: https://drive.google.com/file/d/1IDOadTPN0lM9XnuYjNyJuOwFJRIxFfYQ/view
DATASET_PATH = Path('/content/drive/MyDrive/6ab10eb3b23ba_student_resource.zip')
MODE = 'full'  # 'full': train + all test outputs; 'dev': 3% train check only
RUN_NAME = 'ber_full_v1'  # Keep this to resume. Change for a NEW dataset/settings run.
DENSE_BATCH = 512  # T4; lower to 256 after CUDA out-of-memory, then rerun.
DRIVE_RUN = Path('/content/drive/MyDrive/ML-2-C-26/runs') / RUN_NAME
LOCAL_RUN = Path('/content/ber_runs') / RUN_NAME
REPO_URL = 'https://github.com/SanskariXD/ML-2-C-26.git'
CODE_REF = 'main'  # First run resolves this once; the commit is then locked on Drive.

assert MODE in ('full', 'dev')
assert RUN_NAME and all(c.isalnum() or c in '-_' for c in RUN_NAME), 'Use letters/numbers/-/_ in RUN_NAME'
assert DATASET_PATH.is_file(), f'Drive file not found: {DATASET_PATH}. Mount the account containing the linked ZIP, or adjust its path.'
DRIVE_RUN.mkdir(parents=True, exist_ok=True)
LOCAL_RUN.mkdir(parents=True, exist_ok=True)
print('Dataset:', DATASET_PATH)
print('Outputs and completed-stage backups:', DRIVE_RUN)


## 1. Fetch the code and lock its version
A resumed run uses the same commit. A new RUN_NAME picks up newer code. The repository is public, so no GitHub token is required.


In [ ]:
import os, sys, subprocess, shutil
CODE = Path('/content/ML-2-C-26')
def checked(args, **kwargs):
    return subprocess.run([str(x) for x in args], check=True, **kwargs)
if not (CODE / '.git').exists():
    checked(['git', 'clone', REPO_URL, CODE])
assert subprocess.check_output(['git', '-C', str(CODE), 'remote', 'get-url', 'origin'], text=True).strip() == REPO_URL
assert not subprocess.check_output(['git', '-C', str(CODE), 'status', '--porcelain'], text=True).strip(), 'Local code was edited. Save those edits before switching versions.'
lock = DRIVE_RUN / 'code_commit.txt'
if lock.exists():
    COMMIT = lock.read_text().strip()
else:
    checked(['git', '-C', CODE, 'fetch', 'origin', CODE_REF])
    COMMIT = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
    tmp = lock.with_suffix('.partial')
    tmp.write_text(COMMIT + '\n')
    os.replace(tmp, lock)
checked(['git', '-C', CODE, 'fetch', 'origin', COMMIT])
checked(['git', '-C', CODE, 'checkout', '--detach', COMMIT])
print('Locked commit:', COMMIT)


## 2. Install the isolated environment
The first run downloads Python, CUDA-compatible PyTorch and the pinned dependencies. PyTorch uses the runtime NVIDIA driver; the TPU is unsupported by this pipeline.


In [ ]:
checked([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.8.22'])
UV = shutil.which('uv')
assert UV, 'uv installation failed'
ENV = Path('/content/ber-py311')
PY = ENV / 'bin/python'
if not PY.exists():
    checked([UV, 'python', 'install', '3.11.13'])
    checked([UV, 'venv', '--python', '3.11.13', ENV])
BER = CODE / 'code/business_entity_resolution'
checked([UV, 'pip', 'install', '--python', PY, 'torch==2.7.1',
         '--index-url', 'https://download.pytorch.org/whl/cu126'])
checked([UV, 'pip', 'install', '--python', PY, '-r', BER / 'requirements.txt',
         '-r', BER / 'requirements-dense.txt'])
checked([UV, 'pip', 'check', '--python', PY])
checked([PY, '-c', "import torch, psutil, shutil, sentence_transformers, faiss; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'); print('RAM GiB:', round(psutil.virtual_memory().total/2**30,1)); print('Free disk GiB:', round(shutil.disk_usage('/content').free/2**30,1)); assert torch.cuda.is_available(), 'Choose T4 GPU in runtime settings'"])


## 3. Test installation and copy the existing dataset into runtime storage
The original Drive ZIP stays in place. Only the seven required TSV files are extracted, with checksums. This does not upload or publish your data. The synthetic smoke test checks the CPU pipeline; a small embedding check tests the actual GPU/model download.


In [ ]:
test_env = os.environ.copy()
for key in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS'):
    test_env[key] = '2'
checked([PY, BER / 'tests/smoke_test.py'], env=test_env)
checked([PY, '-c', "from sentence_transformers import SentenceTransformer; m=SentenceTransformer('intfloat/multilingual-e5-small', device='cuda'); x=m.encode(['query: sample business, Bengaluru'], normalize_embeddings=True); assert x.shape==(1,384); print('GPU embedding check PASS')"], env=test_env)
sys.path.insert(0, str(CODE / 'colab'))
from runner import stage_dataset
DATASET = LOCAL_RUN / 'dataset'
stage_dataset(DATASET_PATH, DATASET)
print('Dataset verified:', DATASET)


## 4. Train, predict, validate, and save to Drive
**This cell starts the real run.** Logs stream below. Full mode runs six stages; after each, wait for `SAVED TO DRIVE`. Closing/resetting the runtime before that message can lose the current stage.

If interrupted: reconnect with a T4, keep the same RUN_NAME/MODE and rerun all cells. Complete checkpoints are restored automatically. Partial work from the failed stage is discarded. `dev` mode uses a 3% training subset and never generates a test submission; use a separate RUN_NAME for full mode.


In [ ]:
checked([PY, '-u', CODE / 'colab/runner.py', '--data', DATASET,
         '--local', LOCAL_RUN, '--drive', DRIVE_RUN,
         '--mode', MODE, '--batch', DENSE_BATCH])


## 5. Your files
Leaderboard upload: **matching_results.tsv**. Keep **candidate_pairs.tsv** and the code for the final package. The ZIP here contains output TSVs only; it is not the complete final documentation/code package.


In [ ]:
print('Saved in Google Drive:', DRIVE_RUN)
for name in ['output/matching_results.tsv', 'output/candidate_pairs.tsv', 'ber_submission.zip', 'report.json']:
    p = DRIVE_RUN / name
    if p.exists():
        print(name, f'{p.stat().st_size / 2**20:.1f} MiB')
print('For further details: https://github.com/SanskariXD/ML-2-C-26/blob/main/docs/COLAB_GUIDE.md')
